# Inputs and tools

In the first notebook you handed over a DataFrame, a class and a plotting library, and they
all went in the same way — as keyword arguments.

That is not a coincidence, and it is the idea worth internalising: **JAZ does not distinguish
a prompt from a tool from data.** There is no argument type for any of them, because there
does not need to be. Every keyword argument is an arbitrary Python object that seeds the
agent's REPL under its name and is described in its prompt. One mechanism, and nothing to
configure.

## Setup

Install JAZ, provide an API key, and pick a model. Everything below runs in Colab — no local install.

In [1]:
%pip install -q jaz-lang

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

In [3]:
import jaz
from jaz.llm import LiteLLM

jaz.configure(llm=LiteLLM(model='openai/gpt-5.4-nano'))

## One mechanism, not three

Every keyword argument becomes a variable in the agent's REPL under that name, and gets a
description in the prompt. `task` is not special — it is a string input, and calling it `task`
is a convention, not a rule.

(`ReturnType` is the one thing in these calls that is *not* an input. It is a **hook**, which
is why it goes in positionally rather than by keyword: it constrains the result and never
becomes a variable in the REPL.)

Here is the actual user message for a two-input call. A hook lets us print it.

In [4]:
from jaz import invoke
from jaz.hooks import Hook, ReturnType


class ShowPrompt(Hook):
    """Prints the user message, so we can see exactly what the model was handed."""

    def __init__(self):
        self.shown = False

    def on_llm_query_enter(self, event):
        if not self.shown:
            for message in event.messages:
                if message["role"] == "user":
                    print(message["content"])
                    self.shown = True
                    break
        return []


with ShowPrompt():
    invoke(
        ReturnType(int),
        task="Add the numbers.",
        numbers=[3, 8, 14],
    )

<task type="str">
Add the numbers.
</task>
<numbers type="list">
[3, 8, 14]
</numbers>
The input variables `task`, `numbers` are available in your REPL with descriptions given above.
They are not automatically available to `invoke()` calls you make, so making them available to sub-invokes requires passing them explicitly.



Read that message closely. `task` and `numbers` are listed the same way, each with its type.
Nothing marks one as "the prompt" and the other as "the data" — the agent just gets two named
values.

Note the last line too: inputs are **not** passed on to sub-invokes. That becomes important in
the next notebook.

## A tool is an input that happens to be callable

So a tool needs no registration, no wrapper, no schema. Pass the function.

The example below matters for one reason: **the model cannot answer it on its own.** Stock
levels live in a dict in this notebook. The agent has to call the function to find out — and
notice the task never says which function to call or when. It reads the signature and
docstring and decides.

In [5]:
# A tiny "database". The model has no way to know what is in it.
INVENTORY = {"SKU-1": 12, "SKU-2": 0, "SKU-3": 7, "SKU-4": 0}


def stock_level(sku: str) -> int:
    """Units currently in stock for a SKU. Raises KeyError for an unknown SKU."""
    return INVENTORY[sku]


out_of_stock = invoke(
    ReturnType(list),
    task=(
        "Which of SKU-1, SKU-2, SKU-3, SKU-4 are out of stock? "
        "Return their ids as a list of strings."
    ),
    stock_level=stock_level,
)
out_of_stock

['SKU-2', 'SKU-4']

## How an input is described

Every input is rendered into the prompt, which raises the question the first notebook glossed
over: rendered *how*? It depends on what the object is:

| The input is… | The model is shown |
| :------------ | :----------------- |
| anything else (the default) | its `str()` |
| a class | its docstring and public members |
| any other callable — a function, a method, an object with `__call__` | its signature and docstring |
| an object with no `str()` of its own | its class docstring and public attributes |

You have already seen two rows of that table. `numbers` printed as `[3, 8, 14]` — its `str()`.
`stock_level` went in as a function, so the agent was shown its signature and docstring, which
is the whole reason it could call it correctly without being told how.

The default is not always what you want. For a DataFrame it means every public pandas method
and its docstring — accurate, and enormous:

In [6]:
import pandas as pd

sales = pd.DataFrame(
    {"sku": ["SKU-1", "SKU-2", "SKU-3"], "units": [40, 5, 22]}
)


class MeasurePrompt(Hook):
    """Records how long the user message is."""

    def __init__(self):
        self.chars = None

    def on_llm_query_enter(self, event):
        if self.chars is None:
            for message in event.messages:
                if message["role"] == "user":
                    self.chars = len(message["content"])
                    break
        return []


measure = MeasurePrompt()
with measure:
    invoke(ReturnType(int), task="How many rows does sales have?", sales=sales)

print(f"prompt describing a 3-row DataFrame: {measure.chars:,} characters")

prompt describing a 3-row DataFrame: 50,360 characters


Tens of thousands of characters to say "three rows, two columns". You pay for that in every prompt.

`describe` attaches your own description to an object. It travels with the object, and it replaces the default rendering:

In [7]:
from jaz import describe

described = describe(
    sales,
    "sales: a pandas DataFrame with columns sku (str) and units (int), 3 rows.",
)

measure2 = MeasurePrompt()
with measure2:
    invoke(ReturnType(int), task="How many rows does sales have?", sales=described)

print(f"with a description of our own: {measure2.chars:,} characters")
print(f"saved {measure.chars - measure2.chars:,} characters")

with a description of our own: 400 characters
saved 49,960 characters


Same answer, a fraction of the prompt. Use `describe` whenever the default rendering of an
input is bigger or vaguer than what the agent actually needs.

## What to take away

- **One mechanism.** Prompt, data, tools, classes, libraries — all keyword arguments, all
  arbitrary objects, all variables in the REPL, all described in the prompt. The framework
  draws no distinction between them because it does not need one.
- **Tools need no registration.** A function's signature and docstring *are* its interface.
  Don't tell the agent when to call it; that is its job.
- **Description follows from type.** A value shows its `str()`, a callable its signature and
  docstring, a class its members.
- **You control the rendering.** `describe` is how you decide what an input looks like to the
  model.
- **Hooks are not inputs.** `ReturnType` and friends go in positionally and shape the run;
  they never appear in the agent's REPL.

---

**Next:** [Subagents and scope](03-subagents-and-scope.ipynb) — what happens when an agent
calls `invoke` itself, and why inputs stop at the call that received them.